# Lab 3 — Pipeline Jurídico Completo: Graph RAG + Busca Híbrida + Reranking
## Arquitetura de Produção para Chatbot Investigativo

**Aula 9 | MBA: RAG & CAG Aplicados a Direito e Segurança Pública**

---

### Objetivo
Construir um pipeline de produção que combina:
1. **LightRAG** (Graph RAG) para raciocínio multi-hop;
2. **OpenSearch Hybrid Search** (BM25 + kNN) para recuperação densa+esparsa;
3. **Reranking** com BAAI/bge-reranker para re-ordenar resultados;
4. **Router de intenção** para decidir qual pipeline usar;
5. **Memória conversacional** para contexto de investigação.

## Passo 1: Instalação e Configuração

In [ ]:
# Passo 1.1 — Dependências do pipeline completo
!pip install "lightrag-hku[opensearch]" FlagEmbedding opensearch-py nest-asyncio --quiet

import asyncio
import nest_asyncio
import json
import os
import time
from typing import List, Dict, Optional, Tuple
from enum import Enum

nest_asyncio.apply()
print("✅ Dependências carregadas")

In [ ]:
# Passo 1.2 — Configurações globais do pipeline
from FlagEmbedding import FlagModel, FlagReranker
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from opensearchpy import OpenSearch
import numpy as np

# --- Configurações ---
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "meta-llama/Llama-3.1-8B-Instruct")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "token-vllm")

# --- Carregar modelos locais ---
print("Carregando modelos locais...")
embedding_model = FlagModel('BAAI/bge-m3', use_fp16=True)
print("  ✅ Embedding: BAAI/bge-m3")

# Reranker local
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)
print("  ✅ Reranker: BAAI/bge-reranker-v2-m3")

# --- Cliente OpenSearch ---
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    use_ssl=False,
    verify_certs=False
)
print(f"  ✅ OpenSearch: {OPENSEARCH_HOST}:{OPENSEARCH_PORT}")

print("\n✅ Pipeline pronto!")

## Passo 2: Implementar o Router de Intenção

In [ ]:
# Passo 2.1 — Definir tipos de intenção para roteamento

class TipoIntenção(Enum):
    ENTIDADE_ESPECÍFICA = "entidade_específica"   # "Quem é X?" → Graph RAG local
    MULTI_HOP = "multi_hop"                       # "X se relaciona com Y?" → Graph RAG hybrid
    TEMÁTICO_GLOBAL = "temático_global"           # "Quais os temas?" → Graph RAG global
    BUSCA_LEXICAL = "busca_lexical"               # "Artigo 33 da lei X" → BM25
    BUSCA_SEMÂNTICA = "busca_semântica"           # Busca por significado → kNN
    HÍBRIDA = "híbrida"                           # Combina BM25 + kNN → OpenSearch Hybrid

# Passo 2.2 — Prompt para classificação de intenção
PROMPT_ROUTER = """
Você é um sistema de roteamento de queries jurídicas. Classifique a query em UMA das categorias:

1. ENTIDADE_ESPECÍFICA: pergunta sobre pessoa, lei, tribunal ou processo específico
   Exemplos: "Quem é o réu Silva?", "O que diz o art. 2 da Lei 12.850?"

2. MULTI_HOP: requer conectar múltiplas entidades ou raciocínio em cadeia
   Exemplos: "Como Silva se relaciona com a Operação X?", "Qual a cadeia de prova?"

3. TEMÁTICO_GLOBAL: pergunta sobre padrões, temas gerais ou visão do corpus inteiro
   Exemplos: "Quais são os principais crimes do corpus?", "Que instrumentos são mais usados?"

4. BUSCA_LEXICAL: busca por termos jurídicos exatos, artigos de lei, números de processos
   Exemplos: "HC 127.483", "art. 33 Lei 11.343", "Resolução CNJ 345"

5. BUSCA_SEMÂNTICA: busca por conceito ou significado, sem termos exatos
   Exemplos: "como provar lavagem de dinheiro", "requisitos para colaboração"

6. HÍBRIDA: combina aspectos lexicais e semânticos
   Exemplos: "Lei 12.850 organização criminosa colaboração"

Responda APENAS com o nome da categoria, sem explicações.

Query: {query}
Categoria:"""

print("✅ Router de intenção definido")

In [ ]:
# Passo 2.3 — Implementar função de classificação de intenção

import aiohttp

async def classificar_intenção(query: str) -> TipoIntenção:
    """
    Classifica a intenção da query usando o vLLM.
    Determina qual pipeline de recuperação usar.
    
    Args:
        query: Pergunta do usuário
    
    Returns:
        TipoIntenção correspondente
    """
    prompt = PROMPT_ROUTER.format(query=query)
    
    try:
        # Chamada ao vLLM
        headers = {
            "Authorization": f"Bearer {VLLM_API_KEY}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": VLLM_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 20,
            "temperature": 0.0  # Determinístico
        }
        
        async with aiohttp.ClientSession() as session:
            async with session.post(
                f"{VLLM_BASE_URL}/chat/completions",
                headers=headers, json=payload,
                timeout=aiohttp.ClientTimeout(total=15)
            ) as resp:
                data = await resp.json()
                categoria_str = data['choices'][0]['message']['content'].strip().upper()
        
        # Mapear resposta para enum
        mapeamento = {
            "ENTIDADE_ESPECÍFICA": TipoIntenção.ENTIDADE_ESPECÍFICA,
            "ENTIDADE_ESPECIFICA": TipoIntenção.ENTIDADE_ESPECÍFICA,
            "MULTI_HOP": TipoIntenção.MULTI_HOP,
            "TEMÁTICO_GLOBAL": TipoIntenção.TEMÁTICO_GLOBAL,
            "TEMATICO_GLOBAL": TipoIntenção.TEMÁTICO_GLOBAL,
            "BUSCA_LEXICAL": TipoIntenção.BUSCA_LEXICAL,
            "BUSCA_SEMÂNTICA": TipoIntenção.BUSCA_SEMÂNTICA,
            "BUSCA_SEMANTICA": TipoIntenção.BUSCA_SEMÂNTICA,
            "HÍBRIDA": TipoIntenção.HÍBRIDA,
            "HIBRIDA": TipoIntenção.HÍBRIDA,
        }
        
        for chave, intencao in mapeamento.items():
            if chave in categoria_str:
                return intencao
        
        # Default: busca híbrida
        return TipoIntenção.HÍBRIDA
    
    except Exception as e:
        print(f"  ⚠️ Falha no router LLM ({e}), usando heurística")
        return _classificar_heuristica(query)

def _classificar_heuristica(query: str) -> TipoIntenção:
    """Fallback: classificação por palavras-chave quando o LLM não está disponível."""
    q = query.lower()
    
    # Padrões lexicais
    if any(p in q for p in ['art.', 'artigo', 'lei n', 'hc ', 'resp ', 'resolução', 'n°']):
        return TipoIntenção.BUSCA_LEXICAL
    
    # Padrões multi-hop
    if any(p in q for p in ['relação', 'conecta', 'cadeia', 'como', 'por que', 'quais as conexões']):
        return TipoIntenção.MULTI_HOP
    
    # Padrões globais
    if any(p in q for p in ['principais', 'temas', 'padrões', 'geral', 'corpus', 'todos']):
        return TipoIntenção.TEMÁTICO_GLOBAL
    
    # Padrões de entidade
    if any(p in q for p in ['quem é', 'o que é', 'defina', 'explique']):
        return TipoIntenção.ENTIDADE_ESPECÍFICA
    
    return TipoIntenção.HÍBRIDA

# Testar classificador
queries_teste = [
    "Quem é o réu Marcos Aurélio Silva?",
    "Como a lavagem de dinheiro se relaciona com o tráfico de drogas?",
    "Quais os principais temas do corpus jurídico?",
    "art. 33 da Lei 11.343",
    "requisitos para colaboração premiada",
]

print("Testando classificador heurístico:")
for q in queries_teste:
    intencao = _classificar_heuristica(q)
    print(f"  '{q[:50]}...' → {intencao.value}")

## Passo 3: Implementar Pipeline de Busca Híbrida no OpenSearch

In [ ]:
# Passo 3.1 — Criar índice OpenSearch com campo kNN
# Este índice armazena os chunks do corpus para busca híbrida independente do LightRAG

INDICE_BUSCA = "juridico-busca-hibrida"

MAPPING_HIBRIDO = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 100
        }
    },
    "mappings": {
        "properties": {
            "id": {"type": "keyword"},
            "titulo": {"type": "text", "analyzer": "portuguese"},
            "conteudo": {"type": "text", "analyzer": "portuguese"},
            "tipo_documento": {"type": "keyword"},
            "data": {"type": "date", "format": "yyyy||yyyy-MM-dd"},
            "entidades": {"type": "keyword"},
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "nmslib",
                    "parameters": {"ef_construction": 128, "m": 24}
                }
            }
        }
    }
}

def criar_indice_hibrido():
    """Cria ou recria o índice de busca híbrida no OpenSearch."""
    try:
        # Deletar se existir
        if os_client.indices.exists(index=INDICE_BUSCA):
            os_client.indices.delete(index=INDICE_BUSCA)
            print(f"  🗑️ Índice {INDICE_BUSCA} removido")
        
        # Criar novo
        os_client.indices.create(index=INDICE_BUSCA, body=MAPPING_HIBRIDO)
        print(f"  ✅ Índice {INDICE_BUSCA} criado com sucesso")
        print(f"     Dimensão kNN: 1024 | Analyzer: portuguese | Engine: nmslib/HNSW")
    except Exception as e:
        print(f"  ⚠️ Erro ao criar índice: {e}")

criar_indice_hibrido()

In [ ]:
# Passo 3.2 — Parsear e indexar corpus no OpenSearch com embeddings

import re

def parsear_corpus(caminho: str) -> List[Dict]:
    """Parseia o corpus jurídico em documentos estruturados."""
    with open(caminho, 'r', encoding='utf-8') as f:
        conteudo = f.read()
    
    documentos = []
    blocos = re.split(r'={3}DOCUMENTO_\d+={3}', conteudo)
    
    for bloco in blocos:
        bloco = bloco.strip()
        if not bloco:
            continue
        
        # Extrair título
        titulo_match = re.search(r'TÍTULO:\s*(.+)', bloco)
        titulo = titulo_match.group(1).strip() if titulo_match else "Sem título"
        
        # Extrair tipo
        tipo_match = re.search(r'TIPO:\s*(.+)', bloco)
        tipo = tipo_match.group(1).strip() if tipo_match else "Geral"
        
        # Extrair entidades
        ent_match = re.search(r'ENTIDADES:\s*\[(.+?)\]', bloco, re.DOTALL)
        entidades = []
        if ent_match:
            entidades = [e.strip() for e in ent_match.group(1).split(',')]
        
        # Conteúdo sem metadados
        conteudo_limpo = re.sub(r'(TÍTULO:|TIPO:|DATA:|ENTIDADES:|RELAÇÕES:).*\n', '', bloco)
        conteudo_limpo = conteudo_limpo.strip()
        
        documentos.append({
            "titulo": titulo,
            "tipo_documento": tipo,
            "conteudo": conteudo_limpo,
            "entidades": entidades,
        })
    
    return documentos

documentos = parsear_corpus("../datasets/corpus_juridico.txt")
print(f"✅ {len(documentos)} documentos parseados")
for d in documentos[:3]:
    print(f"  • {d['titulo'][:60]}")

In [ ]:
# Passo 3.3 — Indexar documentos com embeddings no OpenSearch

async def indexar_documentos_opensearch(documentos: List[Dict]):
    """Gera embeddings e indexa documentos no OpenSearch para busca híbrida."""
    print(f"🔄 Indexando {len(documentos)} documentos no OpenSearch...")
    
    # Gerar embeddings em batch
    textos = [f"{d['titulo']}. {d['conteudo'][:500]}" for d in documentos]
    embeddings = await embedding_func_bge(textos)
    
    # Indexar no OpenSearch
    for i, (doc, emb) in enumerate(zip(documentos, embeddings)):
        body = {
            "id": f"doc_{i+1:03d}",
            "titulo": doc["titulo"],
            "conteudo": doc["conteudo"],
            "tipo_documento": doc["tipo_documento"],
            "entidades": doc["entidades"],
            "embedding": emb.tolist()
        }
        
        try:
            os_client.index(index=INDICE_BUSCA, id=f"doc_{i+1:03d}", body=body)
            print(f"  [{i+1}/{len(documentos)}] ✅ {doc['titulo'][:50]}")
        except Exception as e:
            print(f"  [{i+1}] ❌ Erro: {e}")
    
    # Forçar refresh
    os_client.indices.refresh(index=INDICE_BUSCA)
    print(f"\n✅ Indexação concluída!")

asyncio.run(indexar_documentos_opensearch(documentos))

In [ ]:
# Passo 3.4 — Implementar busca híbrida no OpenSearch
# Combina BM25 (busca lexical) com kNN (busca semântica)
# Referência: https://docs.opensearch.org/latest/vector-search/ai-search/hybrid-search/

async def busca_hibrida_opensearch(
    query: str,
    top_k: int = 10,
    peso_bm25: float = 0.3,
    peso_knn: float = 0.7
) -> List[Dict]:
    """
    Realiza busca híbrida no OpenSearch combinando BM25 e kNN.
    
    Args:
        query: Texto da query
        top_k: Número de resultados
        peso_bm25: Peso da busca lexical (0 a 1)
        peso_knn: Peso da busca semântica (0 a 1)
    
    Returns:
        Lista de documentos ordenados por relevância
    """
    # Gerar embedding da query
    embedding_query = await embedding_func_bge([query])
    vetor = embedding_query[0].tolist()
    
    # Query híbrida OpenSearch
    # Usa o hybrid query com normalization processor
    query_body = {
        "query": {
            "hybrid": {
                "queries": [
                    {
                        "match": {
                            "conteudo": {
                                "query": query,
                                "boost": peso_bm25
                            }
                        }
                    },
                    {
                        "knn": {
                            "embedding": {
                                "vector": vetor,
                                "k": top_k
                            }
                        }
                    }
                ]
            }
        },
        "_source": ["id", "titulo", "conteudo", "tipo_documento", "entidades"],
        "size": top_k
    }
    
    try:
        resposta = os_client.search(index=INDICE_BUSCA, body=query_body)
        hits = resposta['hits']['hits']
        
        resultados = []
        for hit in hits:
            resultados.append({
                **hit['_source'],
                'score': hit['_score']
            })
        
        return resultados
    
    except Exception as e:
        print(f"⚠️ Erro na busca híbrida: {e}")
        # Fallback: busca apenas BM25
        query_bm25 = {"query": {"match": {"conteudo": query}}, "size": top_k}
        resposta = os_client.search(index=INDICE_BUSCA, body=query_bm25)
        return [{**h['_source'], 'score': h['_score']} for h in resposta['hits']['hits']]

print("✅ Função de busca híbrida implementada")

## Passo 4: Implementar Reranking com BAAI/bge-reranker

In [ ]:
# Passo 4.1 — Função de reranking
# O reranker avalia a relevância de cada par (query, documento) individualmente
# Isso é mais preciso que o ranking inicial, porém mais custoso

def rerankar_resultados(
    query: str,
    documentos: List[Dict],
    campo_texto: str = "conteudo",
    top_k: int = 5
) -> List[Dict]:
    """
    Reordena documentos usando o modelo BAAI/bge-reranker-v2-m3.
    
    O reranker computa um score de relevância para cada par (query, doc),
    capturando interações semânticas mais finas que a busca inicial.
    
    Args:
        query: Pergunta original do usuário
        documentos: Lista de documentos recuperados
        campo_texto: Campo a usar como conteúdo do documento
        top_k: Número de documentos a retornar após reranking
    
    Returns:
        Top-k documentos reordenados por relevância
    """
    if not documentos:
        return []
    
    # Preparar pares (query, documento) para o reranker
    pares = [
        [query, doc.get(campo_texto, "")[:512]]  # Limitar a 512 tokens
        for doc in documentos
    ]
    
    # Calcular scores de relevância
    scores = reranker.compute_score(pares, normalize=True)
    
    # Garantir que scores seja lista
    if isinstance(scores, float):
        scores = [scores]
    
    # Adicionar score de reranking
    docs_com_score = [
        {**doc, 'rerank_score': float(score)}
        for doc, score in zip(documentos, scores)
    ]
    
    # Ordenar por score de reranking
    docs_ordenados = sorted(docs_com_score, key=lambda x: x['rerank_score'], reverse=True)
    
    return docs_ordenados[:top_k]

# Teste do reranker com dados simulados
query_teste = "colaboração premiada requisitos ministério público"
docs_teste = [
    {"titulo": "Lei 12.850 - Organizações Criminosas",
     "conteudo": "Art. 3 permite colaboração premiada como meio de prova em organizações criminosas"},
    {"titulo": "HC 127.483 STF",
     "conteudo": "A colaboração premiada é negócio jurídico processual personalíssimo celebrado com o MP"},
    {"titulo": "ANPP - Análise",
     "conteudo": "O ANPP difere da colaboração pois não exige fornecimento de informações sobre terceiros"},
    {"titulo": "Manual de Investigação",
     "conteudo": "Ferramentas forenses: Autopsy, Wireshark, Cellebrite para perícia digital"},
]

print("🔄 Testando reranker...")
resultados_reranked = rerankar_resultados(query_teste, docs_teste, top_k=3)
print(f"\nQuery: '{query_teste}'")
print("\nTop-3 após reranking:")
for i, doc in enumerate(resultados_reranked, 1):
    print(f"  [{i}] Score: {doc['rerank_score']:.4f} | {doc['titulo']}")

## Passo 5: Montar o Pipeline Completo com Router

In [ ]:
# Passo 5.1 — Reconectar LightRAG
from lightrag import LightRAG, QueryParam

async def llm_model_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        model=VLLM_MODEL, prompt=prompt,
        system_prompt=system_prompt or "Você é assistente jurídico especializado. Responda em português.",
        history_messages=history_messages,
        api_key=VLLM_API_KEY, base_url=VLLM_BASE_URL, **kwargs
    )

async def embedding_func_bge(texts):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, lambda: embedding_model.encode(texts, batch_size=16))

lightrag = LightRAG(
    llm_model_func=llm_model_func,
    embedding_func=embedding_func_bge,
    embedding_dim=1024,
    kv_storage="OpensearchKVStorage",
    vector_storage="OpensearchVectorDBStorage",
    graph_storage="OpensearchGraphStorage",
    doc_status_storage="JsonDocStatusStorage",
    working_dir="./lightrag_aula9",
    addon_params={
        "opensearch_config": {"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT, "use_ssl": False, "verify_certs": False},
        "language": "Portuguese"
    }
)
print("✅ LightRAG reconectado")

In [ ]:
# Passo 5.2 — Pipeline Jurídico Completo

class PipelineJuridicoCompleto:
    """
    Pipeline de produção para chatbot jurídico investigativo.
    
    Combina:
    - Router de intenção (classifica o tipo de query)
    - LightRAG (Graph RAG para raciocínio multi-hop)
    - Busca híbrida OpenSearch (BM25 + kNN)
    - Reranking com BAAI/bge-reranker
    - Síntese final via vLLM
    """
    
    def __init__(self, lightrag_inst, os_client_inst, reranker_inst):
        self.lightrag = lightrag_inst
        self.os_client = os_client_inst
        self.reranker = reranker_inst
        self.historico: List[Dict] = []
        self.metricas: List[Dict] = []
    
    async def processar_query(
        self,
        query: str,
        usar_historico: bool = True,
        verbose: bool = True
    ) -> Dict:
        """
        Processa uma query através do pipeline completo.
        
        Fluxo:
        1. Classificar intenção
        2. Executar pipeline adequado
        3. Sintetizar resposta final
        4. Atualizar histórico
        
        Args:
            query: Pergunta do usuário
            usar_historico: Incluir histórico da investigação no contexto
            verbose: Exibir detalhes do processamento
        
        Returns:
            Dict com resposta, intenção, pipeline usado e métricas
        """
        inicio_total = time.time()
        
        if verbose:
            print(f"\n{'═'*65}")
            print(f"🔍 Query: {query}")
            print(f"{'─'*65}")
        
        # ETAPA 1: Classificar intenção
        intencao = _classificar_heuristica(query)  # Usar heurística para robustez
        if verbose:
            print(f"📌 Intenção detectada: {intencao.value}")
        
        # ETAPA 2: Executar pipeline adequado
        contexto_graph = ""
        contexto_busca = ""
        pipeline_usado = []
        
        # Pipeline Graph RAG (LightRAG)
        if intencao in [TipoIntenção.ENTIDADE_ESPECÍFICA, TipoIntenção.MULTI_HOP, TipoIntenção.TEMÁTICO_GLOBAL]:
            modo_graph = {
                TipoIntenção.ENTIDADE_ESPECÍFICA: "local",
                TipoIntenção.MULTI_HOP: "hybrid",
                TipoIntenção.TEMÁTICO_GLOBAL: "global"
            }[intencao]
            
            if verbose:
                print(f"🕸️  Executando Graph RAG (modo: {modo_graph})...")
            
            try:
                contexto_graph = await self.lightrag.aquery(
                    query, param=QueryParam(mode=modo_graph)
                )
                pipeline_usado.append(f"graph_rag_{modo_graph}")
                if verbose:
                    print(f"   ✅ Graph RAG retornou {len(contexto_graph)} chars")
            except Exception as e:
                if verbose:
                    print(f"   ⚠️ Graph RAG falhou: {e}")
        
        # Pipeline Busca Híbrida OpenSearch (para todos os tipos)
        if verbose:
            print(f"🔎 Executando busca híbrida OpenSearch...")
        
        try:
            docs_recuperados = await busca_hibrida_opensearch(query, top_k=8)
            
            if docs_recuperados:
                # Reranking
                if verbose:
                    print(f"   📊 Reranking {len(docs_recuperados)} documentos...")
                docs_reranked = rerankar_resultados(query, docs_recuperados, top_k=3)
                pipeline_usado.append("busca_hibrida_reranking")
                
                # Montar contexto de busca
                contextos_busca = []
                for i, doc in enumerate(docs_reranked, 1):
                    contextos_busca.append(
                        f"[Doc {i} — Score: {doc.get('rerank_score', 0):.3f}]\n"
                        f"Título: {doc.get('titulo', '')}\n"
                        f"Conteúdo: {doc.get('conteudo', '')[:300]}..."
                    )
                contexto_busca = "\n\n".join(contextos_busca)
                
                if verbose:
                    print(f"   ✅ Top-3 documentos selecionados após reranking")
        except Exception as e:
            if verbose:
                print(f"   ⚠️ Busca híbrida falhou: {e}")
        
        # ETAPA 3: Síntese final
        if verbose:
            print(f"🧠 Sintetizando resposta final...")
        
        # Montar prompt de síntese
        historico_str = ""
        if usar_historico and self.historico:
            ultimas = self.historico[-3:]  # Últimas 3 interações
            historico_str = "\n".join([
                f"Pergunta anterior: {h['query']}\nResposta: {h['resposta'][:200]}..."
                for h in ultimas
            ])
        
        prompt_sintese = f"""Você é um assistente jurídico especializado em direito penal brasileiro e segurança pública.
Responda à pergunta usando as informações dos contextos fornecidos, de forma precisa e fundamentada.

{f'HISTÓRICO DA INVESTIGAÇÃO:{chr(10)}{historico_str}{chr(10)}' if historico_str else ''}
CONTEXTO DO GRAFO DE CONHECIMENTO:
{contexto_graph if contexto_graph else '(não disponível)'}

DOCUMENTOS RECUPERADOS E RERANKEADOS:
{contexto_busca if contexto_busca else '(não disponível)'}

PERGUNTA: {query}

RESPOSTA JURÍDICA FUNDAMENTADA:"""
        
        try:
            resposta_final = await llm_model_func(prompt_sintese)
        except Exception as e:
            resposta_final = f"[Erro na síntese: {e}. Contexto disponível no grafo e na busca acima.]"
        
        tempo_total = time.time() - inicio_total
        
        # Registrar métricas
        metrica = {
            "query": query[:80],
            "intencao": intencao.value,
            "pipeline": pipeline_usado,
            "tempo_s": round(tempo_total, 2),
            "chars_resposta": len(resposta_final),
            "docs_recuperados": len(docs_recuperados) if 'docs_recuperados' in locals() else 0
        }
        self.metricas.append(metrica)
        
        # Atualizar histórico
        self.historico.append({"query": query, "resposta": resposta_final})
        
        if verbose:
            print(f"\n{'─'*65}")
            print("📋 RESPOSTA FINAL:")
            print(f"{'─'*65}")
            print(resposta_final)
            print(f"\n⏱️ Tempo total: {tempo_total:.2f}s | Pipeline: {' → '.join(pipeline_usado)}")
        
        return {
            "resposta": resposta_final,
            "intencao": intencao.value,
            "pipeline": pipeline_usado,
            "metricas": metrica
        }
    
    def exibir_metricas(self):
        """Exibe relatório de uso do pipeline."""
        if not self.metricas:
            print("Nenhuma query processada ainda.")
            return
        
        import pandas as pd
        df = pd.DataFrame(self.metricas)
        
        print("\n📊 MÉTRICAS DO PIPELINE JURÍDICO")
        print("="*60)
        print(f"Total de queries: {len(df)}")
        print(f"Tempo médio: {df['tempo_s'].mean():.2f}s")
        print(f"\nDistribuição por intenção:")
        print(df['intencao'].value_counts().to_string())
        print(f"\nDetalhes:")
        print(df[['query', 'intencao', 'tempo_s', 'chars_resposta']].to_string(index=False))

# Instanciar o pipeline
pipeline = PipelineJuridicoCompleto(lightrag, os_client, reranker)
print("✅ Pipeline Jurídico Completo instanciado!")

In [ ]:
# Passo 5.3 — Executar sessão de investigação com memória conversacional

async def sessao_investigacao():
    """Simula uma sessão de investigação com múltiplas queries relacionadas."""
    print("🚔 SESSÃO DE INVESTIGAÇÃO — PIPELINE JURÍDICO COMPLETO")
    print("="*65)
    print("Corpus: Documentos jurídicos e de segurança pública")
    print("Stack: LightRAG + OpenSearch Hybrid + bge-reranker-v2-m3 + vLLM")
    print()
    
    # Sequência de queries de investigação (simulam fluxo real de investigador)
    queries_investigacao = [
        "Quais os meios legais de obtenção de prova em organizações criminosas?",
        "Como a prova digital deve ser preservada para ser válida em juízo?",
        "Qual a relação entre colaboração premiada e ANPP no caso em análise?",
        "Quais os principais padrões de lavagem de dinheiro via criptomoedas no corpus?",
    ]
    
    for query in queries_investigacao:
        resultado = await pipeline.processar_query(query, usar_historico=True, verbose=True)
        print()  # Separador

asyncio.run(sessao_investigacao())

In [ ]:
# Passo 5.4 — Exibir métricas da sessão
pipeline.exibir_metricas()

## Passo 6: Análise de Custo-Benefício por Pipeline

### Quando usar cada componente

| Cenário | Pipeline Recomendado | Custo | Qualidade |
|---------|---------------------|-------|----------|
| Identificar conexões entre suspeitos | Graph RAG (hybrid) | Alto | ⭐⭐⭐⭐⭐ |
| Buscar artigo de lei específico | BM25 (lexical) | Baixo | ⭐⭐⭐⭐ |
| Encontrar jurisprudência similar | kNN (semântico) | Médio | ⭐⭐⭐⭐ |
| Resposta geral sobre o corpus | Graph RAG (global) | Alto | ⭐⭐⭐⭐⭐ |
| Chatbot de atendimento jurídico | Híbrido + Reranking | Médio | ⭐⭐⭐⭐ |

### Estimativa de Custo de Indexação

- **Por documento (1000 tokens):** ~10 chamadas ao LLM para extração de entidades
- **Corpus de 1000 docs:** ~10.000 chamadas LLM na indexação
- **Com vLLM local (Llama-3 8B):** custo apenas de energia/GPU
- **Com OpenAI GPT-4o-mini:** ~$0,30 por 1000 docs (estimativa)

> **Recomendação para Segurança Pública:** sempre use vLLM local para indexação de documentos sigilosos.

## Resumo do Lab 3

Você construiu um pipeline de produção completo que:

1. ✅ Classifica automaticamente a intenção da query (router LLM + fallback heurístico)
2. ✅ Executa Graph RAG via LightRAG para raciocínio multi-hop
3. ✅ Combina busca híbrida BM25+kNN no OpenSearch para recuperação precisa
4. ✅ Reordena resultados com BAAI/bge-reranker-v2-m3
5. ✅ Mantém memória conversacional para contexto investigativo
6. ✅ Coleta métricas de uso para análise de performance

**Este é o padrão arquitetural recomendado para chatbots jurídicos e investigativos em produção.**